<a href="https://colab.research.google.com/github/Jeferson03/Trabajo-1/blob/main/notebook/embeddings_bd2_plsql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BD2: Trabajo 1. Punto 2. (Embeddings de Noticias)

## Obtención de las Noticias

Uso de kagglehub para carga del dataset de [BBC News Archive](https://www.kaggle.com/datasets/hgultekin/bbcnewsarchive?select=bbc-news-data.csv). Usamos una submuestra de 200 datos para el proyecto.

In [ ]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "bbc-news-data.csv"

# Load the latest version, skipping bad lines and using the recommended dataset_load
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "hgultekin/bbcnewsarchive",
  file_path,
  pandas_kwargs={'on_bad_lines': 'skip', 'sep': '\t'}
)

print("First 5 records:", df.head())

Using Colab cache for faster access to the 'bbcnewsarchive' dataset.
First 5 records:    category filename                              title  \
0  business  001.txt  Ad sales boost Time Warner profit   
1  business  002.txt   Dollar gains on Greenspan speech   
2  business  003.txt  Yukos unit buyer faces loan claim   
3  business  004.txt  High fuel prices hit BA's profits   
4  business  005.txt  Pernod takeover talk lifts Domecq   

                                             content  
0   Quarterly profits at US media giant TimeWarne...  
1   The dollar has hit its highest level against ...  
2   The owners of embattled Russian oil giant Yuk...  
3   British Airways has blamed high fuel prices f...  
4   Shares in UK drinks and food firm Allied Dome...  


In [ ]:
# Calcular cantidad de palabras en news
news_word_count = df['content'].apply(lambda x: len(x.split()))

news = df[['title','content']].copy()

# Filtrar df para news entre 100 y 500 palabras
news = news[(news_word_count >= 100) & (news_word_count <= 500)]

# submuestra de 200 valores
news = news.sample(200, random_state=1)

news.head()

,title,content
1838,Xbox power cable 'fire fear',Microsoft has said it will replace more than ...
1402,Relay squad thrilled with honours,Jason Gardener says being made an MBE in the ...
334,Boeing secures giant Japan order,Boeing is to supply Japan Airlines with up to...
114,Glaxo aims high after profit fall,GlaxoSmithKline saw its profits fall 9% last ...
1045,Galloway plea for hostage release,Ex-Labour MP George Galloway has appealed for...


## Obtención de Embeddings

Usaremos el model de embeddings **all-MiniLM-L6-v2** con la libreria sentence_transformers de HuggingFace.

all-MiniLM-L6-v2 mapea cada párrado a un vector de 384 dimensiones.

In [ ]:
from sentence_transformers import SentenceTransformer
sentences = news['content'].to_list()
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
embeddings = model.encode(sentences)
print(embeddings)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[[-0.01784443  0.00319523  0.0461257  ... -0.07640739 -0.00851793
  -0.04166698]
 [-0.05354552  0.06205944 -0.0637768  ... -0.08591429 -0.10202339
   0.01938096]
 [ 0.02323253 -0.02074719  0.00428203 ... -0.11589891 -0.00255488
   0.06359156]
 ...
 [-0.02919076 -0.03536924 -0.0613444  ... -0.15861137 -0.06756759
   0.02414251]
 [-0.03348745  0.00297685 -0.03754237 ... -0.08650669  0.00233146
   0.05269076]
 [ 0.03816435  0.08606335  0.02835621 ... -0.08862189 -0.05904713
   0.05777976]]


In [ ]:
import pandas as pd
import numpy as np

# vector a str para escribirlo en la db
embeddings = [emb.tolist() for emb in embeddings]

df_embeddings = pd.DataFrame({"title":news['title'].to_list(), "news":news['content'].to_list(), "embedding_vectors":embeddings})
df_embeddings.head()

,title,news,embedding_vectors
0,Xbox power cable 'fire fear',Microsoft has said it will replace more than ...,"[-0.0178444255143404, 0.003195234341546893, 0...."
1,Relay squad thrilled with honours,Jason Gardener says being made an MBE in the ...,"[-0.0535455159842968, 0.0620594397187233, -0.0..."
2,Boeing secures giant Japan order,Boeing is to supply Japan Airlines with up to...,"[0.02323252707719803, -0.020747192203998566, 0..."
3,Glaxo aims high after profit fall,GlaxoSmithKline saw its profits fall 9% last ...,"[-0.050208184868097305, -0.05553684011101723, ..."
4,Galloway plea for hostage release,Ex-Labour MP George Galloway has appealed for...,"[-0.06966409087181091, -0.02880110964179039, 0..."


## Conexión con la Base de Datos e Inserts con oracledb

In [ ]:
!pip install oracledb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 28.4 MB/s eta 0:00:00


In [ ]:
import oracledb
import os
from google.colab import userdata
user_db = userdata.get('ORACLE_LIVE_USER')
pass_db = userdata.get('ORACLE_LIVE_PASSWORD')

local_dsn = "tcps://db.freesql.com:2484/23ai_34ui2"
connection = oracledb.connect(
    user=user_db,
    password=pass_db,
    dsn=local_dsn)

print("Successfully connected to Oracle Database")



Successfully connected to Oracle Database


In [ ]:
import oracledb
import json

connection = None

try:
    connection = oracledb.connect(
        user=user_db,
        password=pass_db,
        dsn=local_dsn
    )
    print("Re-established connection to Oracle Database for insertion.")

    with connection.cursor() as cursor:
        sql = "INSERT INTO noticia (titulo, texto, embedding) VALUES (:1, :2, :3)"

        # Convert the list of floats in 'embedding_vectors' to a JSON string
        # This makes each embedding a scalar string that Oracle can store.
        data_to_insert = [
            (row['title'], row['news'], json.dumps(row['embedding_vectors']))
            for index, row in df_embeddings.iterrows()
        ]

        cursor.executemany(sql, data_to_insert)

        connection.commit()
    print("Data inserted and committed successfully.")

except Exception as e:
    print(f"An error occurred during insertion: {e}")
    if connection:
        connection.rollback()
    raise

finally:
    if connection:
        connection.close()
        print("Connection closed.")

Re-established connection to Oracle Database for insertion.
Data inserted and committed successfully.
Connection closed.


## Creación de consultas en lenguaje natural y embeddings

In [ ]:
import numpy as np

# Lista de consultas solicitadas
queries = [
    "Show me business news",
    "What's new on Global climate change policy updates?",
    "Advances in renewable energy technology",
    "Has Spain won the Davis cup for the second time in their history?",
    "Show me news about football",
    "Robotics learning and developments",
    "News abouts Argentina",
    "Happy or fun news"
]

# Configuración de visualización de numpy
np.set_printoptions(suppress=True, precision=8)

print(f"Generando embeddings para {len(queries)} consultas...\n")

for i, q in enumerate(queries, 1):
    # Codificar consulta
    query_embedding = model.encode([q])[0]

    # Formatear el vector como string separado por comas
    resultado = ", ".join([f"{x:.10f}" for x in query_embedding])

    print(f"--- Consulta #{i}: {q} ---")
    # Imprimimos los primeros caracteres para verificar el resultado sin saturar la salida
    print(f"{resultado[:150]}... (truncado)")
    print("\n")

Generando embeddings para 8 consultas...

--- Consulta #1: Show me business news ---
0.0358502977, 0.0052922606, 0.0000653336, 0.0455819517, 0.0356294066, 0.0246901661, 0.0301427878, 0.0075091766, 0.0057007372, -0.0249327049, -0.037811... (truncado)


--- Consulta #2: What's new on Global climate change policy updates? ---
-0.0868823081, 0.0104353651, 0.1017336175, 0.0247183181, 0.0972205400, -0.0152494302, -0.0850572810, -0.0370049030, -0.0629340559, 0.0708925873, 0.011... (truncado)


--- Consulta #3: Advances in renewable energy technology ---
-0.0372790359, 0.0953475386, -0.0108548990, -0.0231774319, 0.0091368565, -0.0096842805, -0.0577958636, 0.0638914034, -0.0437103659, 0.0061558178, 0.05... (truncado)


--- Consulta #4: Has Spain won the Davis cup for the second time in their history? ---
0.0085703889, 0.0551324710, -0.0154903205, 0.0097841425, -0.0183904208, 0.0355677083, 0.0172983259, -0.0545240752, 0.0191204064, -0.0052627726, -0.023... (truncado)


--- Consulta #5: Show me n

In [ ]:
import numpy as np

# Template for each embedding insertion
sql_template = """
-- Consulta: {query_text}
DECLARE
  v_clob CLOB := '[{vector_str}]';
BEGIN
  DELETE FROM query_embedding; -- Limpiar para la consulta actual
  INSERT INTO query_embedding VALUES (TO_VECTOR(v_clob));
  COMMIT;
END;
/

SELECT n.idnoticia, n.titulo, n.texto, VECTOR_DISTANCE(n.embedding, q.emb) AS distancia
FROM noticia n, query_embedding q
ORDER BY 3
FETCH FIRST 10 ROWS ONLY;
"""

filename = "queries_embeddings.sql"

with open(filename, "w") as f:
    # 1. Crear la tabla (solo una vez al inicio del archivo)
    f.write("-- 1. Crear la tabla\n")
    f.write("BEGIN EXECUTE IMMEDIATE 'DROP TABLE query_embedding'; EXCEPTION WHEN OTHERS THEN NULL; END;\n/\n")
    f.write("CREATE TABLE query_embedding (emb VECTOR);\n\n")

    print(f"Generando archivo {filename}...")

    for q in queries:
        # Obtener embedding
        embedding = model.encode([q])[0]
        # Convertir a string separado por comas
        vector_str = ", ".join([f"{x:.10f}" for x in embedding])

        # Aplicar plantilla
        block = sql_template.format(query_text=q, vector_str=vector_str)
        f.write(block + "\n")

print(f"Archivo '{filename}' creado exitosamente.")

Generando archivo queries_embeddings.sql...
Archivo 'queries_embeddings.sql' creado exitosamente.


In [ ]:
import numpy as np

accuracy_levels = [90, 70, 50]

sql_template_accuracy = """
-- Consulta: {query_text} | Precisión: {acc}%
DECLARE
  v_clob CLOB := '[{vector_str}]';
BEGIN
  DELETE FROM query_embedding;
  INSERT INTO query_embedding VALUES (TO_VECTOR(v_clob));
  COMMIT;
END;
/

SELECT n.idnoticia, n.titulo, n.texto, VECTOR_DISTANCE(n.embedding, q.emb) AS distancia
FROM noticia n, query_embedding q
ORDER BY VECTOR_DISTANCE(n.embedding, q.emb)
FETCH APPROXIMATE FIRST 10 ROWS ONLY
WITH TARGET ACCURACY {acc} PERCENT;
"""

filename_acc = "queries_accuracy_test.sql"

with open(filename_acc, "w") as f:
    # Setup initial table
    f.write("-- Script de pruebas con TARGET ACCURACY\n")
    f.write("BEGIN EXECUTE IMMEDIATE 'DROP TABLE query_embedding'; EXCEPTION WHEN OTHERS THEN NULL; END;\n/\n")
    f.write("CREATE TABLE query_embedding (emb VECTOR);\n\n")

    print(f"Generando archivo {filename_acc}...")

    for acc in accuracy_levels:
        f.write(f"-- ##############################\n")
        f.write(f"-- PRUEBAS CON ACCURACY: {acc}%\n")
        f.write(f"-- ##############################\n\n")

        for q in queries:
            # Encode query
            embedding = model.encode([q])[0]
            vector_str = ", ".join([f"{x:.10f}" for x in embedding])

            # Apply template for specific accuracy
            block = sql_template_accuracy.format(query_text=q, vector_str=vector_str, acc=acc)
            f.write(block + "\n")

print(f"Archivo '{filename_acc}' creado exitosamente con 3 niveles de precisión para cada query.")

Generando archivo queries_accuracy_test.sql...
Archivo 'queries_accuracy_test.sql' creado exitosamente con 3 niveles de precisión para cada query.
